# Risk indicators — quickstart (live)

The `risk-indicators` backend pulls **country/admin-indexed risk
screening** from three sources behind one interface:

- **GFDRR ThinkHazard!** — a hazard rating (Very low → High) per hazard
  type and admin division; **public**, no key.
- **INFORM Risk (JRC)** — the composite humanitarian-risk index and its
  sub-dimensions, scored 0–10 per country; **public**, no key.
- **Global Forest Watch** Data API — forest indicators and admin geometry;
  needs a free **API key** (covered in the
  [Global Forest Watch notebook](03_global_forest_watch.ipynb)).

Unlike the raster backends there is no bounding box or grid — you name a
dataset id in `variables=` and a country with `country=` (an ISO3 code),
and you get back a tidy `pandas.DataFrame`. This notebook covers the two
**public** sources (ThinkHazard! and INFORM), fetched **live**.

In [1]:
import tempfile

from earthlens.earthlens import EarthLens

OUT = tempfile.mkdtemp()  # tabular downloads also write a CSV here

## ThinkHazard! — one hazard for a country

`variables=["thinkhazard:flood_river"]` selects the river-flood rating;
`country="KEN"` names the country as an ISO3 code. The backend resolves
that ISO3 to ThinkHazard's numeric ADM0 division code and returns one row
with the qualitative `level` / `level_title`.

In [2]:
flood = EarthLens(
    data_source="risk-indicators",
    variables=["thinkhazard:flood_river"],
    country="KEN",
    path=OUT,
).download()

flood

2026-06-27 04:27:27 | INFO | pyramids.base.config | Logging is configured.


2026-06-27 04:27:29.305 | INFO     | earthlens.risk_indicators.backend:_log_citation:419 - RiskIndicators source citation: GFDRR ThinkHazard! — https://thinkhazard.org (public domain).


2026-06-27 04:27:29.310 | INFO     | earthlens.risk_indicators.backend:download:377 - RiskIndicators thinkhazard:flood_river: 1 row(s) written to C:\Users\main\AppData\Local\Temp\tmp062iwcx3\risk_thinkhazard_flood_river.csv.


,country,admin_code,hazard,hazard_type,level,level_title
0,KEN,133,FL,River flood,HIG,High


The `level` is ThinkHazard's coded rating (`VLO`/`LOW`/`MED`/`HIG`) and
`level_title` its human label; `admin_code` is the resolved ADM0 code
(`133` for Kenya). `download()` also wrote the row to a CSV under `path`.

## All eleven hazards at once

The `thinkhazard:all` dataset returns every hazard ThinkHazard! screens
for the division in a single call — handy for a quick risk profile.

In [3]:
profile = EarthLens(
    data_source="risk-indicators",
    variables=["thinkhazard:all"],
    country="KEN",
    path=OUT,
).download()

print("hazards:", len(profile))
profile

2026-06-27 04:27:29.640 | INFO     | earthlens.risk_indicators.backend:_log_citation:419 - RiskIndicators source citation: GFDRR ThinkHazard! — https://thinkhazard.org (public domain).


2026-06-27 04:27:29.643 | INFO     | earthlens.risk_indicators.backend:download:377 - RiskIndicators thinkhazard:all: 11 row(s) written to C:\Users\main\AppData\Local\Temp\tmp062iwcx3\risk_thinkhazard_all.csv.


hazards: 11


,country,admin_code,hazard,hazard_type,level,level_title
0,KEN,133,FL,River flood,HIG,High
1,KEN,133,UF,Urban flood,HIG,High
2,KEN,133,CF,Coastal flood,HIG,High
3,KEN,133,EQ,Earthquake,MED,Medium
4,KEN,133,LS,Landslide,HIG,High
5,KEN,133,TS,Tsunami,MED,Medium
6,KEN,133,VA,Volcano,HIG,High
7,KEN,133,CY,Cyclone,VLO,Very low
8,KEN,133,DG,Water scarcity,MED,Medium
9,KEN,133,EH,Extreme heat,MED,Medium


## INFORM Risk — a composite country score

`inform:risk` is the headline INFORM composite index (0–10). The
`"inform"` alias and the `"risk-indicators"` key both reach the same
backend, so either works as `data_source=`.

In [4]:
risk = EarthLens(
    data_source="inform",
    variables=["inform:risk"],
    country="KEN",
    path=OUT,
).download()

risk

2026-06-27 04:27:29.955 | INFO     | earthlens.risk_indicators.backend:_log_citation:419 - RiskIndicators source citation: INFORM Risk Index, European Commission Joint Research Centre (JRC), Disaster Risk Management Knowledge Centre — https://drmkc.jrc.ec.europa.eu/inform-index.


2026-06-27 04:27:29.958 | INFO     | earthlens.risk_indicators.backend:download:377 - RiskIndicators inform:risk: 1 row(s) written to C:\Users\main\AppData\Local\Temp\tmp062iwcx3\risk_inform_risk.csv.


,iso3,indicator_id,indicator_score,validity_year
0,KEN,INFORM,6.2,0


One row per country: `indicator_score` is the 0–10 composite (higher =
more at risk), `iso3` the country, `validity_year` the model release tag.

**Omit `country=` and INFORM returns every country** — the whole global
table in one frame, ready to rank or join.

In [5]:
world = EarthLens(
    data_source="inform",
    variables=["inform:risk"],
    path=OUT,
).download()

print("countries:", len(world))
world.sort_values("indicator_score", ascending=False).head(5)

2026-06-27 04:27:30.273 | INFO     | earthlens.risk_indicators.backend:_log_citation:419 - RiskIndicators source citation: INFORM Risk Index, European Commission Joint Research Centre (JRC), Disaster Risk Management Knowledge Centre — https://drmkc.jrc.ec.europa.eu/inform-index.


2026-06-27 04:27:30.276 | INFO     | earthlens.risk_indicators.backend:download:377 - RiskIndicators inform:risk: 191 row(s) written to C:\Users\main\AppData\Local\Temp\tmp062iwcx3\risk_inform_risk.csv.


countries: 191


,iso3,indicator_id,indicator_score,validity_year
156,SSD,INFORM,8.5,0
154,SOM,INFORM,8.1,0
187,YEM,INFORM,8.1,0
34,COD,INFORM,8.0,0
165,TCD,INFORM,7.9,0


## The `country=` selector

`country=` is always an **ISO3** code (`"KEN"`, `"NGA"`, `"BRA"`, …).
What happens to it depends on the source:

- **ThinkHazard!** resolves the ISO3 to a GAUL 2015 **ADM0 division
  code** (the numeric key its API uses) via the catalog's bundled
  `admin_codes:` table. For a sub-national division, pass a raw
  `admin_code=` instead.
- **INFORM** filters its global table to that one ISO3 (omit it for all
  countries, as above).
- **GFW** uses the ISO3 as the SQL / geostore key (see the
  [GFW notebook](03_global_forest_watch.ipynb)).

## Takeaway

- `EarthLens(data_source="risk-indicators", variables=[<id>],
  country=<ISO3>)` returns a tidy `DataFrame` — no bbox, no grid.
- ThinkHazard! gives qualitative hazard ratings per division
  (`thinkhazard:all` for the full profile); INFORM gives 0–10 composite
  scores (omit `country=` for the whole world).
- Both are **public** and keyless. The third source, Global Forest Watch,
  needs a free key — see the
  [catalog & behaviour](02_catalog_and_behavior.ipynb) notebook for the
  full dataset list and the [GFW notebook](03_global_forest_watch.ipynb)
  for the keyed forest indicators.